In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Flowsheet-Value/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# 컬럼 삭제
print("\n 컬럼 제거")

columns_to_drop = [
    'cat_text'
]

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        total_cols_to_remove = len(columns_to_drop)
        cols_in_df = [col for col in columns_to_drop if col in df.columns]
        removed_cols_count = len(cols_in_df)
        
        if removed_cols_count > 0:
            df.drop(columns = cols_in_df, inplace = True)
            
        df.to_csv(file, index=False, encoding = 'utf-8')
        
        print(f"{file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")
    
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")

In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt_first ±7일 ±30분
print("\n② hos_dt_first 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt_first' in df.columns:
            df['hos_dt_first'] = pd.to_datetime(df['hos_dt_first'], errors='coerce')
            df['hos_dt_first'] = df['hos_dt_first'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt_first 저장 완료: {file}")
           
    except Exception as e:
        print(f"[❗ hos_dt_first 오류] {file} → {e}")



In [ ]:
# ② cat_dt ±7일 ±30분
print("\n cat_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'cat_dt' in df.columns:
            df['cat_dt'] = pd.to_datetime(df['cat_dt'], errors='coerce')
            df['cat_dt'] = df['cat_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ cat_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ cat_dt 오류] {file} → {e}")



In [ ]:
import numpy as np
# cat_vlaue 처리 (text 제거)
summary = []

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'cat_value' not in df.columns:
            print(f"cat_value 컬럼 미존재 -> {file}")
            continue
            
        total_rows = len(df)
        
        print(f"\n 처리 전 데이터 -> {os.path.basename(file)}")
        print(df.head(3))
        
        # Y,N, 수치형 보존
        keep_mask = (
            df['cat_value'].str.upper().isin(['Y','N']) |
            df['cat_value'].str.match(r'^-?\d+(\.\d+)?$', na=False)
        )
        
        #원본 보존
        original_nonnull = df['cat_value'].notna().sum()
        
        #text 결측치로 대체
        df.loc[~keep_mask, 'cat_value'] = np.nan
        
        replaced_to_na = original_nonnull - df['cat_value'].notna().sum()
        replacement_rate = replaced_to_na / total_rows if total_rows > 0 else 0
        
        print(f"처리 후 샘플 -> {os.path.basename(file)}")
        print(df.head(3))
        
        df.to_csv(file, index = False, encoding = 'utf-8')
        summary.append({
            'file': os.path.basename(file),
            'original_rows': total_rows,
            'replaced_to_NaN': replaced_to_na,
            'replacement_rate(%)': round(replacement_rate * 100, 2)
        })
    
        print(f"처리 완료: {file} --> NaN 대체 {replaced_to_na}건, 비율 {replacement_rate:.2%}")
        
    except Exception as e:
        print(f"오류 발생: {file} -> {e}")
        
summary_df = pd.DataFrame(summary)
print(summary_df.head(10))